In [3]:
# If OpenCV isn't in your env run once:
# !pip install opencv-python-headless

import numpy as np
import cv2 as cv
from pathlib import Path
from IPython.display import HTML, Video


In [4]:
def find_wave_fronts(gray: np.ndarray, k: int = 10) -> np.ndarray:
    """Return k (x,y) points with strongest gradient magnitude."""
    ddepth = cv.CV_16S
    gx = cv.Sobel(gray, ddepth, 1, 0, ksize=3)
    gy = cv.Sobel(gray, ddepth, 0, 1, ksize=3)
    grad = cv.addWeighted(cv.convertScaleAbs(gx), 0.5,
                          cv.convertScaleAbs(gy), 0.5, 0)
    best = np.argsort(grad.ravel())[-k:]
    pts  = np.asarray([np.unravel_index(i, gray.shape)[::-1] for i in best],
                      np.float32)              # (x,y)
    return pts[:, None, :]                     # (k,1,2) for LK

def to_uint8(stack: np.ndarray) -> np.ndarray:
    vmin, vmax = stack.min(), stack.max()
    return (255*(stack - vmin)/(vmax - vmin + 1e-12)).astype(np.uint8)

def save_mp4(frames, path: str, fps=10):
    h, w = frames[0].shape[:2]
    vw = cv.VideoWriter(path,
                        cv.VideoWriter_fourcc(*'mp4v'),
                        fps, (w, h), True)
    for f in frames:
        vw.write(f)
    vw.release()


In [5]:
def track_wave(stack: np.ndarray,
               step: int = 5,
               win_size: int = 100,
               seeds: int = 10,
               out_prefix: str = "wave") -> np.ndarray:
    """
    stack     – ndarray (T,H,W), float or uint8
    returns   – 1-D array of velocities [pixels / frame]
    Saves     – <out_prefix>_tracks.mp4 (overlay video)
    """
    STEP = max(1, step)
    u8   = to_uint8(stack)
    bgr  = [cv.cvtColor(fr, cv.COLOR_GRAY2BGR) for fr in u8]

    # Lucas–Kanade parameters
    lk = dict(winSize=(win_size, win_size),
              maxLevel=1,
              criteria=(cv.TERM_CRITERIA_EPS |
                        cv.TERM_CRITERIA_COUNT, 10, 0.03))

    old_gray = u8[0]
    p0 = find_wave_fronts(old_gray, k=seeds)

    hist   = {i: [(0, tuple(pt[0]))] for i, pt in enumerate(p0)}
    color  = np.random.randint(0, 255, (len(p0), 3))
    mask   = np.zeros_like(bgr[0])
    frames = []

    for f in range(1, len(u8)):
        new_gray = u8[f]
        p1, st, _ = cv.calcOpticalFlowPyrLK(old_gray, new_gray, p0, None, **lk)
        if p1 is None: break

        good_idx = np.where(st == 1)[0]
        good_new = p1[st == 1].reshape(-1, 2)

        fr = bgr[f].copy()
        for idx, (x, y) in zip(good_idx, good_new):
            x_prev, y_prev = hist[idx][-1][1]
            hist[idx].append((f, (x, y)))
            mask = cv.line(mask, (int(x_prev), int(y_prev)),
                                 (int(x),     int(y)),
                                 color[idx].tolist(), 2)
            fr   = cv.circle(fr, (int(x), int(y)), 4,
                             color[idx].tolist(), -1)
        frames.append(cv.add(fr, mask))
        old_gray, p0 = new_gray, p1

    # ------- velocities --------
    vel = []
    for h in hist.values():
        for j in range(STEP, len(h)):
            fn_now,(x_now,y_now)  = h[j]
            fn_prev,(x_prev,y_prev)=h[j-STEP]
            if fn_now - fn_prev == STEP:
                vel.append(np.hypot(x_now-x_prev, y_now-y_prev)/STEP)
    vel = np.asarray(vel)

    # ------- save video --------
    vid_path = f"{out_prefix}_tracks.mp4"
    save_mp4([bgr[0]]+frames, vid_path)
    print(f"Video saved → {vid_path}")
    return vel


In [8]:
stack_path = "michaud_simulation_saved_moving.npy"          # change to your file
stack = np.load(stack_path)            # shape (T,H,W)
print(stack.shape)


(2000, 100, 100)


In [9]:
vel = track_wave(stack,
                 step     = 5,      # distance every 5 frames
                 win_size = 5,
                 seeds    = 10,
                 out_prefix="sim")

print(f"Mean velocity   = {vel.mean():.3f} px/frame")
print(f"Median velocity = {np.median(vel):.3f} px/frame")


Video saved → sim_tracks.mp4
Mean velocity   = 0.076 px/frame
Median velocity = 0.056 px/frame
